In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

### Chargement des Tables de Dimensions

In [0]:
import sys
# -----------------------------
# Ajouter le repo au Python Path
# -----------------------------
sys.path.append("/Workspace/Users/mandu543@gmail.com/databricks_flights")

from src.Communs.utils import *
from conf.config import *

In [0]:
# Param loop Notebook GoldLayer

param_list = [
    {"key_cols_list": "[\"flight_id\"]", "source_object": "silver_flights_flights", "target_object": "dim_flights_flights"},
    {"key_cols_list": "[\"airport_id\"]", "source_object": "silver_flights_airports", "target_object": "dim_flights_airports"},
    {"key_cols_list": "[\"passenger_id\"]", "source_object": "silver_flights_customers", "target_object": "dim_flights_customers"}
]

display(param_list)


In [0]:
for param in param_list:
    dbutils.notebook.run("./03_GoldLayer_02_Dim_opti", 600, param)


### Chargement de la Table de Fait

In [0]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

source_schema = SILVER_ZONE
target_schema = GOLD_ZONE
cdc_col       = "modifiedDate"
surrogate_col = "DimSurrogateKey"

dimensions = [

    {"table": f"{target_schema}.dim_flights_airports", "alias": "airport", "key": "airport_id"},
    {"table": f"{target_schema}.dim_flights_flights",  "alias": "flight",  "key": "flight_id"},
    {"table": f"{target_schema}.dim_flights_customers", "alias": "customer", "key": "passenger_id"}

]

fact_bookings_gold = [

    {

        "source_fact_name" : "silver_flights_bookings",
        "fact_name"        : "fact_flights_bookings",
        "fact_columns"     : ["amount", "booking_date"]

    }

]

In [0]:
# ============================================================
# 2. APPEL DU NOTEBOOK INCREMENTAL
# seul soucis, si on a une nouvelle id pas gérer par le code
# ============================================================

for fact_param in fact_bookings_gold:
    dbutils.notebook.run(
        "03_GoldLayer_03_Fact_opti",   # nom du notebook à appeler
        timeout_seconds=3600,

        arguments={
            "source_schema"    : source_schema,
            "source_table"     : fact_param["source_fact_name"],
            "target_schema"    : target_schema,
            "target_table"     : fact_param["fact_name"],
            "fact_columns"     : ",".join(fact_param["fact_columns"]),  # liste -> string
            "dimensions"       : json.dumps(dimensions),              # dict -> string
            "cdc_col"          : cdc_col,
            "surrogate_col"    : surrogate_col

        }

    )
